<a href="https://colab.research.google.com/github/deltorobarba/quantum/blob/main/qubitization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Qubitization and Quantum Walks in Quantum Computing**

In [2]:
!pip install pennylane -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 52.0 MB/s eta 0:00:00


We have two main steps in this algorithm: Qubitization represents the Hamiltonian (which encodes your problem’s dynamics) in a block-encoded format. Using controlled operations, we can simulate time evolution e⁻ⁱᴴᵗ with fewer resources (fewer qubits and gates). Quantum Walk simulates a walk on a graph or simplicial complex. The quantum walk operator typically combines controlled unitary transformations and phase shifts to explore topological structures efficiently.


**Step 1: Define the Simplicial Complex and Laplacian**

In this step, we’ll define a simplicial complex using an adjacency matrix and construct the Laplacian matrix.


In [3]:
import pennylane as qml
from pennylane import numpy as np

# Define the number of qubits (one per vertex in the simplicial complex)
n_qubits = 3

# Set up a device (let's use default.qubit)
dev = qml.device('default.qubit', wires=n_qubits)

# Example adjacency matrix for a 1-simplex (triangle graph)
adj_matrix = np.array([[0, 1, 1],
                       [1, 0, 1],
                       [1, 1, 0]])

# Define the Laplacian matrix for the graph (simplex)
laplacian_matrix = np.diag(np.sum(adj_matrix, axis=1)) - adj_matrix
print("Laplacian Matrix:\n", laplacian_matrix)

Laplacian Matrix:
 [[ 2 -1 -1]
 [-1  2 -1]
 [-1 -1  2]]


**Step 2: Block Encoding**

PennyLane’s qml.Hermitian observable expects a matrix with dimensions 2ⁿ×2ⁿ, where nn is the number of qubits used in the circuit. In this case, since you have 3 qubits, the Hermitian matrix passed as an observable should be of size 8×8, not 3×3. The current 3×3 Laplacian matrix corresponds to the number of vertices in the simplicial complex, but not the number of qubits.

To resolve this, we “embed” the Laplacian into a larger matrix that matches the qubit system’s dimension, i.e., convert the 3×3 matrix into an 8×8 matrix. We pad the 3×3 Laplacian matrix with zeros to create an 8×8 matrix. By embedding the 3×3 Laplacian matrix into an 8×8 matrix, we ensure that the matrix has the correct dimensions for the 3-qubit system.

Block encoding is a method used to represent a matrix (like a Hamiltonian or Laplacian) using a quantum system, specifically as part of a larger unitary matrix. This allows you to use fewer qubits and gates to simulate time evolution. In this case, we’re trying to represent the Laplacian matrix of a simplicial complex.

In [4]:
# Create an 8x8 matrix by embedding the 3x3 Laplacian into a larger matrix
laplacian_matrix_8x8 = np.zeros((8, 8))  # Initialize a zero matrix of size 8x8
laplacian_matrix_8x8[:3, :3] = laplacian_matrix  # Embed the 3x3 Laplacian in the top-left corner
print("Embedded 8x8 Laplacian Matrix:\n", laplacian_matrix_8x8)

Embedded 8x8 Laplacian Matrix:
 [[ 2. -1. -1.  0.  0.  0.  0.  0.]
 [-1.  2. -1.  0.  0.  0.  0.  0.]
 [-1. -1.  2.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.]]


**Step 3: Qubitization and Quantum Walk**

Next, we block-encode the Laplacian using quantum operations. This involves constructing a controlled unitary that acts on the qubits representing the simplicial complex.

A quantum walk is a process used to explore the structure of a graph (or simplicial complex, in TDA) using quantum operations. The quantum walk evolves the state of the system based on controlled rotations, phase shifts, and other quantum gates. In the context of qubitization, the quantum walk operator helps to simulate the dynamics of a Hamiltonian (or Laplacian in TDA).
The circuit we're constructing in Step 2 applies controlled rotations (RY, CRX) and phase shifts (RZ) to represent the quantum walk over the simplicial complex. These gates simulate the action of the Laplacian on the quantum states, as the quantum walk is applied. This is where we "explore" the topological space (the simplicial complex).

We apply a sequence of gates (like rotations and controlled operations) that simulate how the system evolves (walks) over the simplicial complex, using the block-encoded Laplacian. Rotations and Controlled Gates:

* qml.RY(params[i], wires=i): These rotations represent the quantum walk step by manipulating the state of each qubit.
* qml.CRX(params[0], wires=[0, 1]): This controlled rotation connects different vertices (or simplices) in the simplicial complex, representing transitions in the quantum walk.
* qml.RZ(params[2], wires=0): Phase shifts are added to modify the evolution, often used in quantum walk steps to manipulate the relative phases.

In [5]:
# Quantum walk circuit based on block-encoding the Laplacian
def quantum_walk_circuit(params):
    # Apply block-encoded unitary (for Laplacian) to the qubits
    for i in range(n_qubits):
        qml.RY(params[i], wires=i)  # Using a rotation to represent part of the walk

    # Controlled operation to simulate the quantum walk step
    qml.CRX(params[0], wires=[0, 1])  # Control on qubit 0, rotation on qubit 1
    qml.CRX(params[1], wires=[1, 2])  # Control on qubit 1, rotation on qubit 2

    # You can also add phase shifts or other operations here to simulate the full quantum walk
    qml.RZ(params[2], wires=0)  # Adding a phase shift for completeness

# Create the QNode in PennyLane
@qml.qnode(dev)
def run_quantum_walk(params):
    quantum_walk_circuit(params)
    return qml.state()

# Initialize random parameters for the quantum walk (can be optimized)
params = np.random.random(n_qubits)
state = run_quantum_walk(params)

print("State after quantum walk:", state)

State after quantum walk: [ 0.8580189 -8.44235633e-03j  0.00844236-8.30673781e-05j
  0.41011536-6.23160749e-03j  0.00183936-2.23237417e-01j
  0.18341549-1.99871786e-02j  0.00180469-1.96661034e-04j
  0.0877333 -3.47783490e-02j -0.01780524-4.81822513e-02j]


Here, the quantum_walk_circuit simulates a controlled quantum walk, using RY and CRX gates to apply rotations and controlled operations. These gates act as analogs to the quantum walk steps seen in Qiskit, simulating how the walk would behave on a graph or simplicial complex.

**Step 4: Eigenvalue Estimation and Feature Extraction**

To estimate the eigenvalues of the Laplacian, we can use the block-encoded matrix and measure observables based on the Laplacian. This step allows us to extract topological features, such as eigenvalues related to Betti numbers.

In [6]:
# Define a Hermitian observable using the embedded 8x8 Laplacian matrix
observable = qml.Hermitian(laplacian_matrix_8x8, wires=[0, 1, 2])

@qml.qnode(dev)
def measure_observable(params):
    quantum_walk_circuit(params)
    return qml.expval(observable)

# Measure the observable (eigenvalue estimation)
obs_val = measure_observable(params)
print("Expected value of the observable (Laplacian):", obs_val)

Expected value of the observable (Laplacian): 1.0838516557318096


In this example, we measure the expectation value of the Laplacian matrix after applying the quantum walk. The observable's eigenvalues give us insights into the topological features of the simplicial complex, such as the presence of holes or connected components.

**Step 5: (Optional) Variational Optimization**

We can add an optional optimization step to improve the parameters used in the quantum walk and extract specific features.

In [7]:
# Define a cost function based on the observable
def cost(params):
    return measure_observable(params)

# Initialize the optimizer
opt = qml.GradientDescentOptimizer(stepsize=0.1)

# Optimize the parameters over a few steps
steps = 100
params = np.random.random(n_qubits)
for step in range(steps):
    params = opt.step(cost, params)

# Final value of the observable
print("Final optimized observable value:", cost(params))

Final optimized observable value: 0.061225617944787133


This optimization loop adjusts the parameters of the quantum walk to minimize or maximize the expectation value of the Laplacian matrix, allowing us to extract topological properties more efficiently.